# 26b. 정탐 사례 분석 (True Positive)

모델이 Horizon(30일) 내에 올바르게 고장을 탐지한 사례를 분석합니다.


## 0. 환경 준비


In [1]:
import sys, os, json, joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

sys.path.insert(0, os.path.abspath('..'))

%load_ext autoreload
%autoreload 2

import config.train_config as tcfg
import config.final_eval_config as fe_cfg

warnings.filterwarnings('ignore')

for _font in ['Malgun Gothic', 'NanumGothic', 'AppleGothic', 'DejaVu Sans']:
    if any(_font.lower() in f.name.lower() for f in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = _font
        break
plt.rcParams['axes.unicode_minus'] = False

print('✅ 환경 준비 완료')


✅ 환경 준비 완료


## 1. 모델 및 데이터 로드
08c 와 동일한 단일 임계값(T=0.9590) 기준으로 디스크별 예측 결과를 구성합니다.


In [2]:
from pathlib import Path
from src.eval_core import prepare_disk_level_data
import importlib, src.eval_core
importlib.reload(src.eval_core)

SAVE_DIR = Path(tcfg.MODEL_SAVE_DIR)
BEST_T = 0.9590     # ← 운영 임계값 (08c 기준)
HORIZON = 30        # ← 리드타임 기준 (일)

# ── 모델 로드 ────────────────────────────────────────────────────
with open(SAVE_DIR / 'feature_cols.json', encoding='utf-8') as f:
    FEATURE_COLS = json.load(f)
models = [joblib.load(p) for p in sorted(SAVE_DIR.glob('subset_*.pkl'))]

# ── 테스트 데이터 로드 ────────────────────────────────────────────
test_path = Path(fe_cfg.TEST_PATH)
df_test = pd.read_parquet(test_path)

# ── 예측 확률 (캐시 활용) ─────────────────────────────────────────
cache_filename = f'{test_path.stem}_probs.npy'
cache_path = Path(getattr(tcfg, "PREDICTIONS_DIR", SAVE_DIR)) / cache_filename

if os.path.exists(cache_path):
    print(f'♻️  캐시 로드: {cache_filename}')
    y_prob = np.load(cache_path)
    if len(y_prob) != len(df_test):
        raise ValueError('캐시 크기 불일치. 재예측이 필요합니다.')
else:
    raise FileNotFoundError('예측 캐시가 없습니다. 08b 노트북을 먼저 실행하세요.')

# ── 디스크 단위 데이터 구성 ────────────────────────────────────────
disks_data, n_failed_disks, n_normal_disks = prepare_disk_level_data(
    df_test, y_prob,
    target_col=fe_cfg.TARGET_COL,
    serial_col=fe_cfg.SERIAL_COL,
    date_col=fe_cfg.DATE_COL
)

# ── 08c 와 동일한 로직으로 df_best_disk 생성 ─────────────────────
_records = []
for disk in disks_data:
    probs = disk['probs']
    y_pred = (probs >= BEST_T).astype(int)
    total_alarms = y_pred.sum()
    lead_time = np.nan
    is_alarmed = 0

    if disk['is_failed'] == 1:
        dates = pd.Series(pd.to_datetime(disk['dates']))
        last_date = dates.iloc[-1]
        valid_alarms = (y_pred == 1)
        if valid_alarms.any():
            trigger_idx = np.where(valid_alarms)[0][0]
            trigger_date = dates.iloc[trigger_idx]
            lead_time = (last_date - trigger_date).days
            is_alarmed = 1   # 알람 여부는 모든 케이스를 기록 (집계 시 horizon 필터)
    else:
        if (y_pred == 1).any():
            is_alarmed = 1

    _records.append({
        'base_serial': disk['base_serial'],
        'is_failed': disk['is_failed'],
        'is_alarmed': is_alarmed,
        'total_alarms': total_alarms,
        'lead_time': lead_time,
    })

df_best_disk = pd.DataFrame(_records)

print(f'✅ 데이터 준비 완료 | T={BEST_T} | Horizon={HORIZON}일')
print(f'   총 디스크 수: {len(df_best_disk):,}')
print(f'   고장 디스크: {n_failed_disks:,} | 정상 디스크: {n_normal_disks:,}')


♻️  캐시 로드: test_probs.npy
✅ 데이터 준비 완료 | T=0.959 | Horizon=30일
   총 디스크 수: 7,381
   고장 디스크: 1,135 | 정상 디스크: 6,246


## 2. 정탐(TP) 사례 선정
- **조건**: is_failed=1, is_alarmed=1, lead_time ≤ 30일
- seed=42 로 최대 3개를 자동 샘플링하고 캐시에 저장합니다.


In [3]:
# ────────────────────────────────────────────────────────────────
# 정탐(TP) 사례: is_failed=1, is_alarmed=1, lead_time <= HORIZON
# seed=42 로 3개 샘플링하여 캐시로 저장/로드합니다.
# ────────────────────────────────────────────────────────────────
import random

CACHE_FILE = SAVE_DIR / 'case_selection_09b_tp.json'

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, encoding='utf-8') as f:
        target_serials = json.load(f)
    print(f'♻️  캐시 로드: 정탐 시리얼 {target_serials}')
else:
    tp_pool = df_best_disk[
        (df_best_disk['is_failed'] == 1) &
        (df_best_disk['is_alarmed'] == 1) &
        (df_best_disk['lead_time'] <= HORIZON)
    ]['base_serial'].tolist()

    rng = random.Random(42)
    target_serials = rng.sample(tp_pool, min(3, len(tp_pool)))

    with open(CACHE_FILE, 'w', encoding='utf-8') as f:
        json.dump(target_serials, f, ensure_ascii=False)
    print(f'✅ 정탐(TP) 사례 {len(target_serials)}개 선정: {target_serials}')

# disk_data 딕셔너리로 로드
serial_to_disk = {d['base_serial']: d for d in disks_data}
selected_disks = [serial_to_disk[s] for s in target_serials if s in serial_to_disk]
print(f'로드 완료: {[d["base_serial"] for d in selected_disks]}')


✅ 정탐(TP) 사례 0개 선정: []
로드 완료: []


## 3. 개체 생애 (Lifecycle) 요약


In [4]:

# 누락된 변수 초기화
df_test['base_serial'] = df_test[fe_cfg.SERIAL_COL].str.replace(r'_\d+$', '', regex=True)
df_test['_prob'] = y_prob

import config.interp_config as cfg
BEST_N = 1
ALARM_WINDOW = 14

def print_lifecycle(serial_num, disk_info):
    df_entity = df_test[df_test['base_serial'] == serial_num].sort_values(fe_cfg.DATE_COL)
    first_obs = df_entity[fe_cfg.DATE_COL].min().date()
    last_obs = df_entity[fe_cfg.DATE_COL].max().date()
    
    print(f"\n  [개체 생애 (Lifecycle) 요약: {serial_num}]")
    print(f"  - 데이터 수집 기간 : {first_obs} ~ {last_obs} (총 {len(df_entity)}일간 관측)")
    
    raw_alarms = df_entity[df_entity['_prob'] >= BEST_T]
    
    if disk_info['is_alarmed'] == 1:
        if disk_info['is_failed'] == 1:
            first_alarm = last_obs - pd.Timedelta(days=disk_info['lead_time']) if not np.isnan(disk_info['lead_time']) else "해당 없음"
            print(f"  - 시스템 최종 확정 : {first_alarm} (고장 {disk_info['lead_time']}일 전 감지)")
        else:
            alarms = (df_entity['_prob'] >= BEST_T).astype(int)
            rolling = alarms.rolling(window=ALARM_WINDOW, min_periods=1).sum()
            idx = np.where(rolling >= BEST_N)[0]
            first_alarm_date = df_entity[fe_cfg.DATE_COL].iloc[idx[0]].date() if len(idx) > 0 else "N/A"
            days_before = (last_obs - first_alarm_date).days if len(idx) > 0 else "N/A"
            print(f"  - 시스템 최종 확정 : {first_alarm_date} (알람 발령 후 {days_before}일 동안 고장 없이 정상 동작함)")
    else:
        print(f"  - 시스템 최종 확정 : 없음 (경보 미발령)")
        
    if len(raw_alarms) > 0:
        alarm_details = []
        for d in raw_alarms[fe_cfg.DATE_COL].dt.date.values:
            days_before = (last_obs - d).days
            alarm_details.append(f"D-{days_before}")
        print(f"  - 모델 단일 경고 이력: 총 {len(raw_alarms)}회 ({', '.join(alarm_details)})")
    else:
        print(f"  - 모델 단일 경고 이력: 없음")
        
    if disk_info['is_failed'] == 1:
        status = '✅ 탐지 성공 (Hit)' if disk_info['is_alarmed'] else '❌ 미탐 (Miss)'
        fail_text = '고장'
    else:
        status = '❌ 오탐 (False Alarm)' if disk_info['is_alarmed'] else '✅ 정상 정탐 (True Negative)'
        fail_text = '정상'
        
    print(f"  - 최종 평가 결과   : {status} ({fail_text} 디스크)\n")

for serial in target_serials:
    info = df_best_disk[df_best_disk['base_serial'] == serial].iloc[0]
    print("="*60)
    print_lifecycle(serial, info)


## 4. Waterfall Plot (최대 위험 시점 피처 기여도)


In [5]:
import shap
for serial in target_serials:
    df_entity = df_test[df_test['base_serial'] == serial].sort_values(fe_cfg.DATE_COL)
    X_entity = df_entity[FEATURE_COLS].reset_index(drop=True)

    # 앙상블 전체의 SHAP 평균 계산
    sv_entity_list = []
    for model in models:
        exp = shap.TreeExplainer(model)
        sv_e = exp.shap_values(X_entity)
        if isinstance(sv_e, list):
            sv_e = sv_e[1]
        sv_entity_list.append(sv_e)
    mean_sv_entity = np.mean(sv_entity_list, axis=0)

    # 고장 확률이 가장 높은 시점 선택
    peak_idx = df_entity["_prob"].values.argmax()
    peak_date = df_entity[fe_cfg.DATE_COL].iloc[peak_idx].date()
    peak_prob = df_entity["_prob"].iloc[peak_idx]
    
    print(f"\n" + "="*70)
    print(f"[§9.2] Waterfall Plot — 개체: {serial}")
    print(f"       최고 위험 시점: {peak_date} (예측 확률={peak_prob:.4f})")
    print("="*70)

    # 첫 번째 모델 기준 expected_value 사용
    base_vals = np.mean([shap.TreeExplainer(m).expected_value for m in models])
    if isinstance(base_vals, np.ndarray):
        base_vals = base_vals[1]

    # data=None으로 설정하여 좌측의 연한 회색 원본 피처값을 숨김
    explanation = shap.Explanation(
        values=mean_sv_entity[peak_idx],
        base_values=float(base_vals),
        feature_names=FEATURE_COLS,
    )

    shap.plots.waterfall(explanation, max_display=15, show=False)
    
    # 마이너스 기호 깨짐 해결
    fig = plt.gcf()
    for ax in fig.axes:
        for t in ax.texts:
            t.set_text(t.get_text().replace('\u2212', '-'))
        labels = [l.get_text().replace('\u2212', '-') for l in ax.get_xticklabels()]
        ax.set_xticklabels(labels)
        labels = [l.get_text().replace('\u2212', '-') for l in ax.get_yticklabels()]
        ax.set_yticklabels(labels)
        
    fig.suptitle(f"최대 고장 위험 시점의 피처별 SHAP 기여도 분석 (대상 개체: {serial})", fontsize=14, fontweight="bold", y=1.05)
    plt.tight_layout()
    plt.show()


## 5. 시간 기반 SHAP Trajectory (고장 전 N일)


In [6]:
window_days = cfg.TEMPORAL_WINDOW_DAYS
top_n = cfg.TEMPORAL_TOP_N_FEATS

for serial in target_serials:
    df_entity = df_test[df_test['base_serial'] == serial].sort_values(fe_cfg.DATE_COL)
    info = df_best_disk[df_best_disk['base_serial'] == serial].iloc[0]
    
    is_failed = info['is_failed']
    is_alarmed = info['is_alarmed']
    
    alarms = (df_entity['_prob'] >= BEST_T).astype(int)
    rolling = alarms.rolling(window=ALARM_WINDOW, min_periods=1).sum()
    idx = np.where(rolling >= BEST_N)[0]
    first_alarm_date = df_entity[fe_cfg.DATE_COL].iloc[idx[0]] if len(idx) > 0 else None
    
    if is_failed == 1:
        ref_date = df_entity[fe_cfg.DATE_COL].max()
        title_suffix = "Failure Date (D-0)"
    else:
        if is_alarmed == 1:
            if first_alarm_date is not None:
                ref_date = min(first_alarm_date + pd.Timedelta(days=5), df_entity[fe_cfg.DATE_COL].max())
            else:
                ref_date = df_entity[fe_cfg.DATE_COL].max()
            title_suffix = "Observation End (D-0)"
        else:
            ref_date = df_entity[fe_cfg.DATE_COL].max()
            title_suffix = "Observation End (D-0)"

    start_date = ref_date - pd.Timedelta(days=window_days - 1)
    df_window = df_entity[(df_entity[fe_cfg.DATE_COL] >= start_date) & (df_entity[fe_cfg.DATE_COL] <= ref_date)].copy().reset_index(drop=True)
    
    print(f"\n\n" + "="*70)
    print(f"[§9.3] 시간 기반 SHAP Trajectory — 개체: {serial}")
    print(f"       분석 기간: {start_date.date()} ~ {ref_date.date()} ({len(df_window)}일)")
    print("="*70)

    if len(df_window) == 0:
        print("⚠️  분석 기간 내 데이터가 없습니다. TEMPORAL_WINDOW_DAYS를 늘려보세요.")
        continue

    X_window = df_window[FEATURE_COLS].reset_index(drop=True)

    sv_window_list = []
    for model in models:
        exp = shap.TreeExplainer(model)
        sv_w = exp.shap_values(X_window)
        if isinstance(sv_w, list):
            sv_w = sv_w[1]
        sv_window_list.append(sv_w)
    mean_sv_window = np.mean(sv_window_list, axis=0)

    max_pos_window = np.max(mean_sv_window, axis=0) 
    top_feat_idx = np.argsort(max_pos_window)[::-1][:top_n]
    top_feat_names = [FEATURE_COLS[idx] for idx in top_feat_idx]

    dates = df_window[fe_cfg.DATE_COL].dt.date.values
    days_from_last = [- (ref_date.date() - d).days for d in dates]

    fig, axes = plt.subplots(top_n + 1, 1, figsize=(12, 3 * (top_n + 1)), sharex=True)
    fig.patch.set_facecolor('#ffffff')
    
    ax_prob = axes[0]
    probs = df_window["_prob"].values
    
    ax_prob.plot(days_from_last, probs, marker='o', color='#2c3e50', lw=2.5, markersize=5, label="Prediction Probability")
    ax_prob.axhline(BEST_T, color='#e74c3c', linestyle='--', lw=2, label=f"Threshold (T={BEST_T:.3f})")
    ax_prob.fill_between(days_from_last, probs, BEST_T, where=(probs >= BEST_T), color='#e74c3c', alpha=0.15)
    
    if first_alarm_date is not None:
        alarm_day = - (ref_date.date() - first_alarm_date.date()).days
        if -window_days <= alarm_day <= 0:
            ax_prob.axvline(alarm_day, color='#e74c3c', linestyle=':', lw=2.5, alpha=0.8)

    ax_prob.set_ylabel("Probability", fontsize=11, fontweight="bold", color='#34495e')
    ax_prob.set_title("Model Prediction Probability", fontsize=13, fontweight="bold", loc='left', color='#2c3e50')
    ax_prob.set_ylim(-0.05, 1.05)
    ax_prob.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='#ecf0f1')
    ax_prob.grid(axis='y', linestyle='--', alpha=0.4)
    ax_prob.spines['top'].set_visible(False)
    ax_prob.spines['right'].set_visible(False)
    ax_prob.spines['left'].set_color('#bdc3c7')
    ax_prob.spines['bottom'].set_color('#bdc3c7')

    pos_color = '#e6614f'
    neg_color = '#4a90e2'
    
    for ax, feat_name, feat_idx in zip(axes[1:], top_feat_names, top_feat_idx):
        shap_vals = mean_sv_window[:, feat_idx]
        bars = ax.bar(days_from_last, shap_vals, 
                      color=[pos_color if v > 0 else neg_color for v in shap_vals], 
                      alpha=0.85, width=0.8)
        ax.axhline(0, color="#7f8c8d", lw=1.2, ls="-")
        
        if first_alarm_date is not None:
            alarm_day = - (ref_date.date() - first_alarm_date.date()).days
            if -window_days <= alarm_day <= 0:
                ax.axvline(alarm_day, color='#e74c3c', linestyle=':', lw=2.5, alpha=0.5, zorder=0)

        max_abs = max(abs(shap_vals.min()), abs(shap_vals.max())) * 1.1
        if max_abs > 0:
            ax.set_ylim(-max_abs, max_abs)
            
        ax.set_ylabel("SHAP Value", fontsize=11, fontweight="bold", color='#34495e')
        ax.set_title(f"Feature: {feat_name}", fontsize=12, fontweight="bold", loc='left', color='#2c3e50')
        ax.grid(axis="y", linestyle='--', alpha=0.4)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#bdc3c7')
        ax.spines['bottom'].set_color('#bdc3c7')

    tick_locs = np.arange(-window_days + 1, 1, 5)
    if 0 not in tick_locs:
        tick_locs = np.append(tick_locs, 0)
    
    xticklabels = []
    for x in tick_locs:
        if x in days_from_last:
            idx = days_from_last.index(x)
            real_date = dates[idx].strftime('%m-%d')
        else:
            real_date = (ref_date + pd.Timedelta(days=int(x))).strftime('%m-%d')
        xticklabels.append(f"D{int(x)}\n({real_date})")

    axes[-1].set_xticks(tick_locs)
    axes[-1].set_xticklabels(xticklabels, fontsize=10, color='#34495e')
    axes[-1].set_xlabel(f"Timeline to {title_suffix}", fontsize=12, fontweight="bold", color='#2c3e50', labelpad=15)

    fig.suptitle(
        f"Temporal SHAP Evolution & Risk Trajectory (Serial: {serial})",
        fontsize=16, fontweight="bold", color='#2c3e50', y=1.02
    )
    plt.tight_layout()
    plt.show()
    print(f"\n개체 {serial} 분석 완료. 상위 {top_n}개 피처: {top_feat_names}")


## 6. Compact Temporal SHAP Trajectory (논문 본문용)
- 정보량이 많아 세로로 길어지는 기존 플롯 대신, 논문 본문에 삽입하기 좋은 **Compact Version**입니다.
- 핵심 피처 3개만 표시하며, Probability 패널의 비중을 높여 시각적 안정감을 줍니다.

In [7]:
window_days = cfg.TEMPORAL_WINDOW_DAYS
top_n = 3  # 논문용 압축 버전

for serial in target_serials:
    df_entity = df_test[df_test['base_serial'] == serial].sort_values(fe_cfg.DATE_COL)
    info = df_best_disk[df_best_disk['base_serial'] == serial].iloc[0]
    
    is_failed = info['is_failed']
    is_alarmed = info['is_alarmed']
    
    alarms = (df_entity['_prob'] >= BEST_T).astype(int)
    rolling = alarms.rolling(window=ALARM_WINDOW, min_periods=1).sum()
    idx = np.where(rolling >= BEST_N)[0]
    first_alarm_date = df_entity[fe_cfg.DATE_COL].iloc[idx[0]] if len(idx) > 0 else None
    
    if is_failed == 1:
        ref_date = df_entity[fe_cfg.DATE_COL].max()
        title_suffix = "Failure Date (D-0)"
    else:
        if is_alarmed == 1:
            if first_alarm_date is not None:
                ref_date = min(first_alarm_date + pd.Timedelta(days=5), df_entity[fe_cfg.DATE_COL].max())
            else:
                ref_date = df_entity[fe_cfg.DATE_COL].max()
            title_suffix = "Observation End (D-0)"
        else:
            ref_date = df_entity[fe_cfg.DATE_COL].max()
            title_suffix = "Observation End (D-0)"

    start_date = ref_date - pd.Timedelta(days=window_days - 1)
    df_window = df_entity[(df_entity[fe_cfg.DATE_COL] >= start_date) & (df_entity[fe_cfg.DATE_COL] <= ref_date)].copy().reset_index(drop=True)
    
    print(f"\n\n" + "="*70)
    print(f"[§9.3 Compact] 시간 기반 SHAP Trajectory — 개체: {serial}")
    print(f"       분석 기간: {start_date.date()} ~ {ref_date.date()} ({len(df_window)}일)")
    print("="*70)

    if len(df_window) == 0:
        print("⚠️  분석 기간 내 데이터가 없습니다. TEMPORAL_WINDOW_DAYS를 늘려보세요.")
        continue

    X_window = df_window[FEATURE_COLS].reset_index(drop=True)

    sv_window_list = []
    for model in models:
        exp = shap.TreeExplainer(model)
        sv_w = exp.shap_values(X_window)
        if isinstance(sv_w, list):
            sv_w = sv_w[1]
        sv_window_list.append(sv_w)
    mean_sv_window = np.mean(sv_window_list, axis=0)

    max_pos_window = np.max(mean_sv_window, axis=0) 
    top_feat_idx = np.argsort(max_pos_window)[::-1][:top_n]
    top_feat_names = [FEATURE_COLS[idx] for idx in top_feat_idx]

    dates = df_window[fe_cfg.DATE_COL].dt.date.values
    days_from_last = [- (ref_date.date() - d).days for d in dates]

    fig, axes = plt.subplots(top_n + 1, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [2.5] + [1]*top_n})
    fig.patch.set_facecolor('#ffffff')
    
    ax_prob = axes[0]
    probs = df_window["_prob"].values
    
    ax_prob.plot(days_from_last, probs, marker='o', color='#2c3e50', lw=2.5, markersize=5, label="Prediction Probability")
    ax_prob.axhline(BEST_T, color='#e74c3c', linestyle='--', lw=2, label=f"Threshold (T={BEST_T:.3f})")
    ax_prob.fill_between(days_from_last, probs, BEST_T, where=(probs >= BEST_T), color='#e74c3c', alpha=0.15)
    
    if first_alarm_date is not None:
        alarm_day = - (ref_date.date() - first_alarm_date.date()).days
        if -window_days <= alarm_day <= 0:
            ax_prob.axvline(alarm_day, color='#e74c3c', linestyle=':', lw=2.5, alpha=0.8)

    ax_prob.set_ylabel("Probability", fontsize=11, fontweight="bold", color='#34495e')
    ax_prob.set_title("Model Prediction Probability", fontsize=13, fontweight="bold", loc='left', color='#2c3e50')
    ax_prob.set_ylim(-0.05, 1.05)
    ax_prob.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='#ecf0f1')
    ax_prob.grid(axis='y', linestyle='--', alpha=0.4)
    ax_prob.spines['top'].set_visible(False)
    ax_prob.spines['right'].set_visible(False)
    ax_prob.spines['left'].set_color('#bdc3c7')
    ax_prob.spines['bottom'].set_color('#bdc3c7')

    pos_color = '#e6614f'
    neg_color = '#4a90e2'
    
    for ax, feat_name, feat_idx in zip(axes[1:], top_feat_names, top_feat_idx):
        shap_vals = mean_sv_window[:, feat_idx]
        bars = ax.bar(days_from_last, shap_vals, 
                      color=[pos_color if v > 0 else neg_color for v in shap_vals], 
                      alpha=0.85, width=0.8)
        ax.axhline(0, color="#7f8c8d", lw=1.2, ls="-")
        
        if first_alarm_date is not None:
            alarm_day = - (ref_date.date() - first_alarm_date.date()).days
            if -window_days <= alarm_day <= 0:
                ax.axvline(alarm_day, color='#e74c3c', linestyle=':', lw=2.5, alpha=0.5, zorder=0)

        max_abs = max(abs(shap_vals.min()), abs(shap_vals.max())) * 1.1
        if max_abs > 0:
            ax.set_ylim(-max_abs, max_abs)
            
        ax.set_ylabel("SHAP Value", fontsize=11, fontweight="bold", color='#34495e')
        ax.set_title(f"Feature: {feat_name}", fontsize=12, fontweight="bold", loc='left', color='#2c3e50')
        ax.grid(axis="y", linestyle='--', alpha=0.4)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#bdc3c7')
        ax.spines['bottom'].set_color('#bdc3c7')

    tick_locs = np.arange(-window_days + 1, 1, 5)
    if 0 not in tick_locs:
        tick_locs = np.append(tick_locs, 0)
    
    xticklabels = []
    for x in tick_locs:
        if x in days_from_last:
            idx = days_from_last.index(x)
            real_date = dates[idx].strftime('%m-%d')
        else:
            real_date = (ref_date + pd.Timedelta(days=int(x))).strftime('%m-%d')
        xticklabels.append(f"D{int(x)}\n({real_date})")

    axes[-1].set_xticks(tick_locs)
    axes[-1].set_xticklabels(xticklabels, fontsize=10, color='#34495e')
    axes[-1].set_xlabel(f"Timeline to {title_suffix}", fontsize=12, fontweight="bold", color='#2c3e50', labelpad=15)

    fig.suptitle(
        f"Temporal SHAP Evolution & Risk Trajectory (Serial: {serial})",
        fontsize=16, fontweight="bold", color='#2c3e50', y=1.02
    )
    plt.tight_layout()
    plt.show()
    print(f"\n개체 {serial} 분석 완료. 상위 {top_n}개 피처: {top_feat_names}")
